In [ ]:
%load_ext memprofiler
import numba
import numpy as np

import blosc2

In [ ]:
# os.environ["BLOSC_BLOCKSIZE"] = str(128 * 1024)

In [ ]:
# For best speed
blosc2.cparams_dflts["codec"] = blosc2.Codec.BLOSCLZ
# blosc2.cparams_dflts["codec"] = blosc2.Codec.LZ4
# blosc2.cparams_dflts["codec"] = blosc2.Codec.ZSTD
blosc2.cparams_dflts["clevel"] = 1
# blosc2.cparams_dflts["filters"] = [blosc2.Filter.BITSHUFFLE]
# blosc2.cparams_dflts["filters_meta"] = [0]

# blosc2.nthreads = 16
# blosc2.cparams_dflts["nthreads"] = blosc2.nthreads
# blosc2.dparams_dflts["nthreads"] = blosc2.nthreads
# ne.set_num_threads(blosc2.nthreads)  # ensure a fair comparison with numexpr
# numba.set_num_threads(blosc2.nthreads)  # ensure a fair comparison with numba

In [ ]:
%%time
# N = 35_000
N = 20_000
na = np.linspace(0, 1, N * N).reshape(N, N)
nb = np.linspace(1, 2, N * N).reshape(N, N)
nc = np.linspace(-10, 10, N * N).reshape(N, N)

In [ ]:
%%time
# Convert to blosc2
a = blosc2.asarray(na)
b = blosc2.asarray(nb)
c = blosc2.asarray(nc)

In [ ]:
%%time
# Expression (blosc2 form)
# expr = (a * 2 + b > c)
# expr = ((a ** 3 + blosc2.sin(c * 2)) < b)
expr = ((a**3 + blosc2.sin(c * 2)) < b) & (c > 0)
# numexpr form
# sexpr = "(na * 2 + nb > nc)"
# sexpr = "((na ** 3 + sin(nc * 2)) < nb)"
sexpr = "((na ** 3 + sin(nc * 2)) < nb) & (nc > 0)"

In [ ]:
# %%mprof_run 0.lazyexpr::mmap-warmup
# # Warm memory-map cache
# out = expr.compute()

In [ ]:
%%mprof_run 1.lazyexpr::compute-BLOSCLZ-1
# compute and get a NDArray as result
out = expr.compute()

In [ ]:
out.info

In [ ]:
%%mprof_run 1.lazyexpr::getitem-BLOSCLZ-1
# compute and get a NDArray as result
out_ = expr[:]

In [ ]:
%%mprof_run 2.NumExpr
# compute with numexpr
out1 = ne.evaluate(sexpr)

In [ ]:
@numba.jit(parallel=True)
def func_expr(inputs_tuple, output, offset):
    a = inputs_tuple[0]
    b = inputs_tuple[1]
    c = inputs_tuple[2]
    for i in numba.prange(a.shape[0]):
        for j in numba.prange(a.shape[1]):
            # expr = (a[i, j] * 2 + b[i, j] > c[i, j])
            # expr = ((a[i, j] ** 3 + np.sin(c[i, j] * 2)) < b[i, j])
            expr = ((a[i, j] ** 3 + np.sin(c[i, j] * 2)) < b[i, j]) and (c[i, j] > 0)
            output[i, j] = expr
    output[:] = expr


lzyudf = blosc2.lazyudf(func_expr, (a, b, c), np.bool_)

In [ ]:
%%mprof_run 3.Numba
out2 = np.empty(out.shape, dtype=out.dtype)
func_expr((na, nb, nc), out2, 0)

In [ ]:
%%time
blosc2.cparams_dflts["clevel"] = 0
a = blosc2.asarray(na)
b = blosc2.asarray(nb)
c = blosc2.asarray(nc)
expr = ((a**3 + blosc2.sin(c * 2)) < b) & (c > 0)

In [ ]:
%%mprof_run 4.lazyexpr::compute-nocompr
# compute and get a NDArray as result
out3 = expr.compute()

In [ ]:
out3.info

In [ ]:
%%mprof_run 4.lazyexpr::getitem-nocompr
# compute and get a NDArray as result
out3_ = expr[:]

In [ ]:
%%mprof_run 5.NumPy
# compute with numpy
#out = (na * 2 + nb > nc) & (nc > 0)
#out = ((na ** 3 + np.sin(nc * 2)) < nb)
out = ((na ** 3 + np.sin(nc * 2)) < nb) & (nc > 0)

In [ ]:
%mprof_plot .* -t "AMD 7950X3D -- Number of threads: {blosc2.nthreads}"